# Module 04 — Assignment: AlexNet's toolkit

Module 03's assignment built the eye; this one builds what AlexNet needed to
make the eye work at scale. You implement four load-bearing pieces — the
**strided, padded convolution**, **overlapping max-pool's gradient routing**,
**inverted dropout**, and **crop + flip augmentation** — and the check cells
grade you against finite differences and the canonical `python/alexnet.py` +
`nanograd` answer key. The finale drops your pieces into a GIVEN harness that
must reproduce the module's toy result: memorize 32 corner glyphs facing one
way, then recognize their **mirror images**.

Rules: don't peek at `lib/python/nanograd` or `python/alexnet.py` until you've
tried. The check cells are your feedback. Fill every `# TODO`, run top to bottom.

In [ ]:
# Setup — run once. (GIVEN: do not modify.)
import os, sys, math
import numpy as np

def _find_repo():
    here = os.getcwd()
    cands = [here] + [os.path.abspath(os.path.join(here, *(['..'] * k))) for k in range(1, 6)]
    for base in cands:
        if os.path.isdir(os.path.join(base, 'lib', 'python', 'nanograd')):
            return base
    return here

REPO = _find_repo()
TOPIC = os.path.join(REPO, 'topics', '04-alexnet')
sys.path.insert(0, os.path.join(REPO, 'lib', 'python'))
sys.path.insert(0, os.path.join(TOPIC, 'python'))
sys.path.insert(0, os.path.join(TOPIC, 'tests'))
import nanograd as ng
import alexnet as canon
from check_utils import (rel_error, eval_numerical_gradient_array, compare_to_canonical)
from nanograd.rng import Rng
print('setup ok — nanograd and check utils imported')

## Part 1 — the strided, padded convolution

Stride $S$ moves the window $S$ pixels at a time; padding surrounds the input
with $P_{\text{pad}}$ rings of zeros. One output-size formula covers both:

$$H_{\text{out}} = \left\lfloor \frac{H + 2P_{\text{pad}} - K}{S} \right\rfloor + 1$$

and output cell $(u, v)$ reads the input at an offset:

$$Y[i,f,u,v] = b_f + \sum_{c,p,q} K[f,c,p,q]\, X[i,c,\,uS{+}p{-}P_{\text{pad}},\,vS{+}q{-}P_{\text{pad}}]$$

where a tap that lands in the padding reads 0 — so just skip it. Write the
output size, then the inner accumulation (the outer loops are GIVEN).

In [ ]:
def my_conv_out_size(H, K, stride, pad):
    out = None
    ###########################################################################
    # TODO: the output-size formula above, in integer arithmetic. 1 line.
    ###########################################################################
    # out = ...
    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
    return out


def my_conv2d_forward(X, K, b, stride=1, pad=0):
    n, C, H, W = X.shape
    F, _, KH, KW = K.shape
    OH, OW = my_conv_out_size(H, KH, stride, pad), my_conv_out_size(W, KW, stride, pad)
    Y = np.empty((n, F, OH, OW))
    for i in range(n):                      # GIVEN: the canonical loop nest
        for f in range(F):
            for u in range(OH):
                for v in range(OW):
                    acc = None
                    ###########################################################################
                    # TODO: start acc at b[f]; for every c, p, q compute the input row
                    # h = u*stride + p - pad and column w = v*stride + q - pad, skip the
                    # tap if (h, w) is outside the image, else add K[f,c,p,q] * X[i,c,h,w].
                    # ~7 lines.
                    ###########################################################################
                    # acc = ...
                    # for c in range(C):
                    #     ...
                    ###########################################################################
                    #                          END OF YOUR CODE
                    ###########################################################################
                    Y[i, f, u, v] = acc
    return Y

In [ ]:
# Check: your forward vs the canonical definition AND the vectorized nanograd
# layer, for three stride/padding combinations.
rng = np.random.default_rng(0)
assert my_conv_out_size(227, 11, 4, 0) == 55, 'output-size formula is off'
for stride, pad, H in ((1, 0, 6), (2, 2, 9), (3, 1, 8)):
    X = rng.standard_normal((2, 2, H, H)); K = rng.standard_normal((3, 2, 3, 3)); b = rng.standard_normal(3)
    Y_you = my_conv2d_forward(X, K, b, stride, pad)
    conv = ng.Conv2D(2, 3, 3, 3, stride=stride, pad=pad); conv.K[:] = K; conv.b[:] = b
    d_naive = np.abs(Y_you - canon.naive_conv2d_forward(X, K, b, stride, pad)).max()
    d_fast = np.abs(Y_you - conv.forward(X)).max()
    print(f'S={stride} pad={pad}: out {Y_you.shape[2:]}  max|diff| vs naive {d_naive:.1e}, vs nanograd {d_fast:.1e}')
    assert d_naive < 1e-12 and d_fast < 1e-12
compare_to_canonical([float(Y_you.sum()), float(Y_you[1, 2, 0, 1])],
                     [float(conv.forward(X).sum()), float(conv.forward(X)[1, 2, 0, 1])],
                     labels=['sum', 'Y[1,2,0,1]'])

### Inline Question 1
AlexNet's first layer slides 11×11 kernels over a 227×227 image at stride 4,
no padding. How big is its output map, and roughly how many times fewer
positions does it evaluate than the same kernel at stride 1? Why would anyone
choose that trade at the *first* layer rather than a later one? *Your answer:* ____

## Part 2 — overlapping max-pool: routing gradients to winners

AlexNet pools 3×3 windows every 2 pixels, so neighboring windows share cells.
The forward pass (GIVEN) records, for every window, the flat index into $X$ of
the cell that won. The backward pass routes each window's gradient to its
winner. The subtlety: one cell can win **two** windows — and a value used
twice receives **both** gradients, added.

Careful: NumPy's `dX[idx] += dY` with repeated indices does **not**
accumulate — each repeated index is written once, and the other gradients are
silently lost. `np.add.at(dX, idx, dY)` does accumulate.

In [ ]:
class MyMaxPool:
    def __init__(self, P=3, stride=2):
        self.P, self.S = P, stride

    def forward(self, X):
        # GIVEN: max over each window; remember the winner's flat index into X
        # (first maximum in row-major window order, like the library and the C).
        n, C, H, W = X.shape
        P, S = self.P, self.S
        OH, OW = (H - P) // S + 1, (W - P) // S + 1
        self.x_shape = X.shape
        Y = np.empty((n, C, OH, OW))
        self.idx = np.empty((n, C, OH, OW), dtype=np.int64)
        plane = (np.arange(n * C) * H * W).reshape(n, C)       # where each (i, c) image starts
        for u in range(OH):
            for v in range(OW):
                win = X[:, :, u * S:u * S + P, v * S:v * S + P].reshape(n, C, P * P)
                k = win.argmax(axis=-1)
                Y[:, :, u, v] = np.take_along_axis(win, k[..., None], axis=-1)[..., 0]
                self.idx[:, :, u, v] = plane + (u * S + k // P) * W + (v * S + k % P)
        return Y

    def backward(self, dY):
        dX = np.zeros(int(np.prod(self.x_shape)))
        ###########################################################################
        # TODO: scatter-ADD every dY value into dX at its window's winner,
        # self.idx (same shape as dY). Use np.add.at so a cell that won
        # several windows collects all of their gradients. 1 line.
        ###########################################################################
        # np.add.at(...)
        ###########################################################################
        #                          END OF YOUR CODE
        ###########################################################################
        return dX.reshape(self.x_shape)

In [ ]:
# Check 1: a cell that wins every window collects every gradient.
x = np.zeros((1, 1, 5, 5)); x[0, 0, 2, 2] = 9.0
pool = MyMaxPool(); pool.forward(x)
dx = pool.backward(np.ones((1, 1, 2, 2)))
print('center cell won all 4 windows; its gradient =', dx[0, 0, 2, 2], '(should be 4.0)')
assert dx[0, 0, 2, 2] == 4.0, 'overlapping windows must ADD their gradients (np.add.at)'

# Check 2: finite differences, and the canonical nanograd layer.
Xp = np.random.default_rng(3).standard_normal((2, 3, 7, 7))
pool = MyMaxPool(); Yp = pool.forward(Xp)
dYp = np.random.default_rng(4).standard_normal(Yp.shape)
dX_you = pool.backward(dYp)
num = eval_numerical_gradient_array(lambda _: MyMaxPool().forward(Xp), Xp, dYp)
print(f'dX vs finite differences: rel_error = {rel_error(dX_you, num):.2e}')
assert rel_error(dX_you, num) < 1e-6
ref = ng.MaxPool2D(3, 2); ref.forward(Xp)
compare_to_canonical([float(dX_you.sum()), float((dX_you * np.arange(dX_you.size).reshape(dX_you.shape)).sum())],
                     [float(ref.backward(dYp).sum()), float((ref.backward(dYp) * np.arange(dX_you.size).reshape(dX_you.shape)).sum())],
                     labels=['dX_sum', 'dX_wsum'])

### Inline Question 2
Suppose you had written `dX[self.idx.reshape(-1)] += dY.reshape(-1)`. Which
cells of $dX$ would come out wrong, and would the finite-difference check
always catch it? (Hint: when do two windows share a winner?) *Your answer:* ____

## Part 3 — inverted dropout

During training each unit survives with probability $1-p$, and survivors are
scaled up so the expected activation is unchanged:

$$m_j = \frac{[\,u_j < 1-p\,]}{1-p},\qquad y = x \odot m, \qquad \frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \odot m$$

Draw **all** the $u_j$ with one call, `self.rng.random(X.shape)` (that's what
makes your masks identical to the library's when both use the same `Rng`). In
evaluation mode the layer is the identity.

In [ ]:
class MyDropout:
    def __init__(self, p, rng):
        self.p, self.rng = p, rng
        self.training = True
        self.mask = None

    def forward(self, X):
        if not self.training:               # GIVEN: evaluation is the identity
            self.mask = None
            return X
        out = None
        ###########################################################################
        # TODO: keep = 1 - p; draw u = self.rng.random(X.shape); set
        # self.mask = (u < keep) / keep (as floats); out = X * self.mask. ~3 lines.
        ###########################################################################
        # keep = ...
        # self.mask = ...
        # out = ...
        ###########################################################################
        #                          END OF YOUR CODE
        ###########################################################################
        return out

    def backward(self, dY):
        dX = None
        ###########################################################################
        # TODO: a dropped unit sent nothing forward, so it gets nothing back.
        # In eval mode (mask None) the gradient passes straight through. ~1-2 lines.
        ###########################################################################
        # dX = ...
        ###########################################################################
        #                          END OF YOUR CODE
        ###########################################################################
        return dX

In [ ]:
# Check 1: bit-identical to the library when both draw from the same LCG.
X = np.random.default_rng(5).standard_normal((4, 10))
you, lib = MyDropout(0.5, Rng(7)), ng.Dropout(0.5, rng=Rng(7))
Y_you, Y_lib = you.forward(X), lib.forward(X)
print('mask values:', np.unique(you.mask), '  identical to nanograd.Dropout:', np.array_equal(Y_you, Y_lib))
assert np.array_equal(Y_you, Y_lib)

# Check 2: the expectation is preserved (p = 0.25 this time).
big = MyDropout(0.25, np.random.default_rng(0)).forward(np.ones((50000, 4)))
print('mean activation over 50,000 draws (should be ~1):', np.round(big.mean(axis=0), 3))
assert np.allclose(big.mean(axis=0), 1.0, atol=0.02)

# Check 3: backward vs finite differences with the mask held fixed.
d = MyDropout(0.5, None)
def fwd(_):
    d.rng = Rng(11)
    return d.forward(Xd)
Xd = np.random.default_rng(6).standard_normal((3, 8)); dYd = np.random.default_rng(7).standard_normal((3, 8))
fwd(None); dX_you = d.backward(dYd)
err = rel_error(dX_you, eval_numerical_gradient_array(fwd, Xd, dYd))
print(f'dX vs finite differences: rel_error = {err:.2e}'); assert err < 1e-6
d.training = False
assert np.array_equal(d.forward(Xd), Xd) and np.array_equal(d.backward(dYd), dYd), 'eval mode must be the identity'
ref = ng.Dropout(0.5, rng=Rng(11)); ref.forward(Xd)
compare_to_canonical([float(dX_you.sum())], [float(ref.backward(dYd).sum())], labels=['dX_sum'])

### Inline Question 3
The 2012 paper did *not* scale survivors during training; instead it
multiplied every hidden output by 0.5 at test time. Show that the two
conventions give the same expected activation. What would go wrong if you did
**neither**? *Your answer:* ____

## Part 4 — crops and mirrors

For each image draw three uniforms, `u = rng.random((n, 3))` (GIVEN): a row
offset $dy = \lfloor u_0 (H - \text{crop} + 1) \rfloor$, a column offset
$dx = \lfloor u_1 (W - \text{crop} + 1) \rfloor$, and a mirror if $u_2 < 0.5$.
Copy the crop×crop window at $(dy, dx)$, reversed left-to-right when mirrored.

In [ ]:
def my_random_crop_flip(X, crop, rng):
    n, C, H, W = X.shape
    u = rng.random((n, 3))                  # GIVEN: row offset, col offset, flip
    out = np.empty((n, C, crop, crop), dtype=X.dtype)
    for i in range(n):
        ###########################################################################
        # TODO: compute dy and dx (int(...) of the formulas above), slice the
        # patch X[i, :, dy:dy+crop, dx:dx+crop], and store it in out[i] --
        # reversed along the last axis (patch[:, :, ::-1]) if u[i, 2] < 0.5. ~5 lines.
        ###########################################################################
        # dy = ...
        # ...
        pass
        ###########################################################################
        #                          END OF YOUR CODE
        ###########################################################################
    return out

In [ ]:
# Check: identical to nanograd's random_crop_flip from the same LCG stream --
# which in turn is what c/alexnet.c draws.
Xg, _ = canon.make_glyphs(8, canon.DATA_SEED)
mine = my_random_crop_flip(Xg, canon.CROP, Rng(canon.AUG_SEED))
lib = ng.random_crop_flip(Xg, canon.CROP, Rng(canon.AUG_SEED))
print('shape', mine.shape, '  identical to the library:', np.array_equal(mine, lib))
assert mine.shape == (8, 3, 10, 10) and np.array_equal(mine, lib)
compare_to_canonical([float(mine.sum()), float(mine[3, 1].sum())], [float(lib.sum()), float(lib[3, 1].sum())],
                     labels=['sum', 'img3_ch1'])

### Inline Question 4
In the module's toy, training on crops + flips raised accuracy on the
*mirrored* test glyphs from ~69% to ~95%, but dropout alone did nothing. Why
can't dropout do what the flip did? Where in AlexNet *does* dropout earn its
keep, and why there? *Your answer:* ____

## Verify: your pieces reproduce the module's toy

A GIVEN harness builds the toy AlexNet (the library's strided `Conv2D` and
`Linear` layers) and trains it for 600 SGD steps using **your** max-pool
backward, **your** dropout, and **your** crops and flips, all drawing from the
same LCG streams as `c/alexnet.c`. Your network must memorize the 32 training
glyphs and recognize the mirrored test glyphs as well as the canonical run
does (`run_toy()`'s fingerprint).

In [ ]:
def train_toy_with(pool_cls, dropout_cls, crop_flip, steps=canon.TOY_STEPS):
    '''GIVEN harness: the toy AlexNet trained with YOUR pieces. Do not modify.'''
    Xtr, ytr = canon.make_glyphs(canon.N_TRAIN, canon.DATA_SEED)
    Xte, yte = canon.make_glyphs(canon.N_TEST, canon.TEST_SEED, mirrored=True)
    r = Rng(canon.TOY_SEED)
    conv = ng.Conv2D(3, canon.TOY_F, canon.TOY_KS, canon.TOY_KS, stride=canon.TOY_S, pad=canon.TOY_PAD)
    fc1, fc2 = ng.Linear(canon.N_FLAT, canon.N_HID), ng.Linear(canon.N_HID, 2)
    ng.he_normal_conv2d(conv.K, r); ng.he_normal(fc1.W, r); ng.he_normal(fc2.W, r)
    pool, drop, aug = pool_cls(), dropout_cls(canon.DROP_P, Rng(canon.DROP_SEED)), Rng(canon.AUG_SEED)
    lossf = ng.SoftmaxCrossEntropy()
    opt = ng.SGD(lr=canon.SGD_LR, momentum=canon.SGD_MOM, weight_decay=canon.SGD_WD)

    def forward(X, train):
        drop.training = train
        Z = conv.forward(X); A = np.maximum(Z, 0.0); P = pool.forward(A)
        H1 = fc1.forward(P.reshape(len(X), -1)); Hr = np.maximum(H1, 0.0)
        return fc2.forward(drop.forward(Hr)), (Z, P, H1)

    for _ in range(steps):
        logits, (Z, P, H1) = forward(crop_flip(Xtr, canon.CROP, aug), True)
        lossf.forward(logits, ytr)
        dH = drop.backward(fc2.backward(lossf.backward())) * (H1 > 0.0)
        dA = pool.backward(fc1.backward(dH).reshape(P.shape)) * (Z > 0.0)
        conv.backward(dA)
        opt.step([conv.K, conv.b, fc1.W, fc1.b, fc2.W, fc2.b],
                 [conv.dK, conv.db, fc1.dW, fc1.db, fc2.dW, fc2.db])
    acc = lambda X, y: float((forward(ng.center_crop(X, canon.CROP), False)[0].argmax(1) == y).mean())
    return acc(Xtr, ytr), acc(Xte, yte)

train_acc, test_acc = train_toy_with(MyMaxPool, MyDropout, my_random_crop_flip)
print(f'trained with YOUR pieces:  train acc = {train_acc:.3f}   mirrored test acc = {test_acc:.3f}')
canon_fp = dict(canon.run_toy())
compare_to_canonical([train_acc, test_acc], [canon_fp['acc'], canon_fp['test_acc']],
                     labels=['acc', 'test_acc'])
print('OK — strided conv, overlapping pooling, dropout, and augmentation all pull together.')